In [1]:
import torch
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenizing Dataset
def tokenize(text):
  text=text.lower()
  text=text.replace("?","")
  text=text.replace("'","")
  return text.split()

tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [4]:
# vocab
vocab={'<UNK>':0}
def build_vocab(row):
  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])
  merged_tokens=tokenized_question+tokenized_answer
  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)

In [5]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [6]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [7]:
# convert words to numerical indices
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token not in vocab:
      indexed_text.append(vocab['<UNK>'])
    else:
      indexed_text.append(vocab[token])
  return indexed_text


In [8]:
text_to_indices("What is the age of Kabir?",vocab)

[1, 2, 3, 0, 5, 0]

In [10]:
len(vocab)

324

In [11]:
from torch.utils.data import DataLoader , Dataset


In [14]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df=df
    self.vocab=vocab
  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question), torch.tensor(numerical_answer)




In [15]:
dataset=QADataset(df,vocab)

In [16]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [17]:
data_loader=DataLoader(dataset,batch_size=1,shuffle=True)

In [18]:
import torch.nn as nn

In [27]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size):

    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True) # batch_first=True removes unneccesary dimensionality [1,6,324] which is caused by the swapping internally in RNN
    self.fc=nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question=self.embedding(question)
    hidden , final =self.rnn(embedded_question)
    output=self.fc(final.squeeze(0)) # to remove the batch dimension

    return output


In [28]:
learning_rate=0.001
epochs=20

In [29]:
model=SimpleRNN(len(vocab))
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=learning_rate)

In [31]:
# training loop
for epoch in range(epochs):
  total_loss=0
  for question, answer in data_loader:
    optimizer.zero_grad()
    # forward pass
    output=model(question)
    # loss output.shape=(1,324) whereas answer.shape=(1,)
    loss=criterion(output,answer[0]) # answer is  a 2d matrix
    # gradient
    loss.backward()
    #update
    optimizer.step()

    total_loss=total_loss+loss.item()
  print(f"Epoch: {epoch+1} , Loss:  {total_loss:4f}")

Epoch: 1 , Loss:  523.356928
Epoch: 2 , Loss:  457.571901
Epoch: 3 , Loss:  378.011371
Epoch: 4 , Loss:  314.339560
Epoch: 5 , Loss:  261.097718
Epoch: 6 , Loss:  211.504889
Epoch: 7 , Loss:  167.160549
Epoch: 8 , Loss:  128.642476
Epoch: 9 , Loss:  97.640926
Epoch: 10 , Loss:  73.822584
Epoch: 11 , Loss:  56.309527
Epoch: 12 , Loss:  43.946298
Epoch: 13 , Loss:  34.680036
Epoch: 14 , Loss:  27.974029
Epoch: 15 , Loss:  22.850711
Epoch: 16 , Loss:  19.171405
Epoch: 17 , Loss:  16.102993
Epoch: 18 , Loss:  13.730263
Epoch: 19 , Loss:  11.750746
Epoch: 20 , Loss:  10.211465


In [43]:
def predict(model, question, threshold=0.5):
  #convert question into numbers
  numerical_question=text_to_indices(question,vocab)

  # tensor
  question_tensor=torch.tensor(numerical_question).unsqueeze(0)

  #send to model
  output=model(question_tensor)

  #converts logits to  probs
  probs=torch.nn.functional.softmax(output,dim=1)
  #find max prob
  value , index =torch.max(probs,dim=1)
  if value < threshold:
    print("I don't know")
  else:
    print(list(vocab.keys())[index])

In [44]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [46]:
predict(model,"What is the capital of france?")

paris
